# Markout analysis — per-client flow toxicity
For every client fill, the **markout at horizon h** is the client-signed mid move after the fill:
$$m_h = s \cdot (mid_{t+h} - mid_t), \quad s = +1\ \text{for BUY}, -1\ \text{for SELL}$$
- **Positive** markout: the market moved the client's way after they traded — informed / toxic flow; the counterparty (us, when internalized) is adversely selected.
- **Negative** markout: price mean-reverted against the client's trade — benign / noise flow; inventory taken from them tends to profit (the theoretical basis of positive drift P&L).

Horizons: 30s / 2min / 10min, share-weighted per client. Fills whose horizon extends past the last quote are dropped at that horizon (no end-of-day clamping bias). One engineered day — treat rankings as illustrative, not statistically significant.

In [ ]:
import sys
from bisect import bisect_right
from datetime import datetime, timedelta

import pandas as pd

sys.path.insert(0, '.')
import config
from internalizer.data import load_quotes

quotes = load_quotes(config.QUOTES_CSV)
qts = [q.ts for q in quotes]
mids = [(q.bid + q.ask) / 2 for q in quotes]
LAST = qts[-1]

def mid_at(ts):
    """Prevailing mid (cents) at or before ts."""
    return mids[bisect_right(qts, ts) - 1]

fills = pd.read_csv(config.FILLS_CSV, parse_dates=['timestamp'])
print(f'{len(fills)} fills, {fills.client_id.nunique()} clients')
fills.head(3)

In [ ]:
HORIZONS = {'30s': 30, '2min': 120, '10min': 600}

rows = []
for _, f in fills.iterrows():
    s = 1 if f['side'] == 'BUY' else -1          # client-signed direction
    m0 = mid_at(f['timestamp'])                  # mid at fill time
    row = {'client': f['client_id'].replace('CLIENT_', ''), 'qty': f['quantity'],
           'venue': f['venue']}
    for name, secs in HORIZONS.items():
        t1 = f['timestamp'] + timedelta(seconds=secs)
        # drop fills whose horizon runs past the tape (avoids clamping bias)
        row[name] = s * (mid_at(t1) - m0) if t1 <= LAST else None
    rows.append(row)
mo = pd.DataFrame(rows)

def share_weighted(g):
    out = {'fills': len(g), 'shares': g['qty'].sum()}
    for h in HORIZONS:
        v = g.dropna(subset=[h])
        out[f'markout {h} (c/sh)'] = round((v[h] * v['qty']).sum() / v['qty'].sum(), 2)
    return pd.Series(out)

per_client = (mo.groupby('client').apply(share_weighted, include_groups=False)
                .sort_values('markout 10min (c/sh)', ascending=False))
per_client   # positive at the top = most toxic (market moves their way after they trade)

In [ ]:
# aggregate flow character + internalized-only view (what the firm actually absorbed)
total = share_weighted(mo)
internal = share_weighted(mo[mo.venue == 'INTERNAL'])
pd.DataFrame({'all fills': total, 'internalized only': internal})

**How to read this for the strategy:**
- Clients at the top (positive 10-min markout) are candidates for a **toxicity multiplier** on the 2¢ internalization threshold, or for routing outright — internalizing their flow means holding inventory that trends against us.
- Clients at the bottom (negative markout) are the benign flow whose mean reversion is the theoretical source of positive inventory-drift P&L.
- If `internalized only` markouts are worse than `all fills`, our acceptance rule is adversely selecting toxic flow into the book; if better, the spread/coverage gates are already filtering well.
- Caveat: one engineered day, a handful of fills per client — production tiering needs weeks of data and horizon-matched significance tests.

---
## Markout-adjusted edge — the risk-adjusted metric

Everything above is **client-signed** (positive = toxic flow). The firm holds the
opposite side, so **firm P&L = −(client-signed markout)**. This double negative is the
classic sign-error trap in markout work, so from here on both conventions are named
explicitly in the code (`client_signed` / `firm_signed`) rather than left to a comment.

For each internalized fill the firm's economics have two parts:

$$\text{adjusted edge}_h \;=\; \underbrace{(mid_t - px)\cdot q_{firm}}_{\text{execution edge}} \;+\; \underbrace{q_{firm}\cdot(mid_{t+h} - mid_t)}_{\text{firm markout}}$$

- **execution edge**: what the fill earned against fair value at that instant (0 on even
  spreads where mid is a whole cent, +0.5¢ on odd spreads where rounding favours the firm).
- **firm markout**: what the inventory taken on would have been worth h later, if held.

Reads only `fills.csv` + the quote tape, so any version's output can be analysed
without re-running the engine: point `FILLS_PATH` at `out/fills.csv`, a saved
`out/fills_v0.csv`, etc. `nbbo_bid`/`nbbo_ask` are recorded on every fill row, so the
spread bucket and the fair value at fill time both come straight from the CSV.

**Caveat on horizon choice**: markout assumes each fill's full size is held in isolation
for h. The book actually nets 228,300 gross internalized shares down to ~1,559 shares of
time-weighted exposure (146× turnover), so long horizons overstate what the firm really
carried. Match the horizon to realised holding time; the 1-minute column is the honest
one here.

In [ ]:
FILLS_PATH = config.FILLS_CSV        # any version's fills.csv works; no engine replay
ADJ_HORIZONS = [('5s', 5), ('30s', 30), ('1min', 60), ('5min', 300), ('30min', 1800)]


def markout_adjusted_edge(fills_path=FILLS_PATH, horizons=ADJ_HORIZONS):
    """Per-spread-bucket execution edge and firm markout, straight from a fills CSV.

    Sign convention, spelled out once: `client_signed` is +qty for a client BUY;
    the firm takes the other side, so `firm_signed = -client_signed`. Firm markout
    is firm_signed x (mid_future - mid_fill): a short book gains when the mid falls.
    """
    df = pd.read_csv(fills_path, parse_dates=['timestamp'])
    df = df[df['venue'] == 'INTERNAL'].copy()          # principal risk only

    px = (df['price'] * 100).round()                    # cents
    bid = (df['nbbo_bid'] * 100).round()
    ask = (df['nbbo_ask'] * 100).round()
    df['spread_c'] = (ask - bid).astype(int)
    df['mid'] = (bid + ask) / 2
    df['client_signed'] = df['quantity'].where(df['side'] == 'BUY', -df['quantity'])
    df['firm_signed'] = -df['client_signed']
    df['edge'] = (df['mid'] - px) * df['firm_signed']   # cents x shares, firm view

    for name, secs in horizons:                         # firm markout at each horizon
        t1 = df['timestamp'] + pd.Timedelta(seconds=secs)
        future_mid = [mid_at(t) if t <= LAST else None for t in t1]
        df[f'mo_{name}'] = [None if m is None else fs * (m - m0)
                            for m, fs, m0 in zip(future_mid, df['firm_signed'], df['mid'])]

    rows = {}
    for spr, g in df.groupby('spread_c'):
        r = {'shares': int(g['quantity'].sum()),
             'edge (c/sh)': round(g['edge'].sum() / g['quantity'].sum(), 2)}
        for name, _ in horizons:                        # drop fills whose horizon runs off the tape
            v = g.dropna(subset=[f'mo_{name}'])
            r[f'adj {name}'] = (round((v['edge'].sum() + v[f'mo_{name}'].sum())
                                      / v['quantity'].sum(), 2) if len(v) else None)
        rows[f'{spr}c'] = r
    out = pd.DataFrame(rows).T

    all_r = {'shares': int(df['quantity'].sum()),
             'edge (c/sh)': round(df['edge'].sum() / df['quantity'].sum(), 2)}
    for name, _ in horizons:
        v = df.dropna(subset=[f'mo_{name}'])
        all_r[f'adj {name}'] = round((v['edge'].sum() + v[f'mo_{name}'].sum())
                                     / v['quantity'].sum(), 2)
    out.loc['ALL'] = all_r
    return out, df


adjusted, internal_fills = markout_adjusted_edge()
adjusted   # cents/share, firm view: execution edge + inventory markout at each horizon

In [ ]:
# dollar view: how little of the value is execution edge once inventory is marked
dollars = []
for name, _ in ADJ_HORIZONS:
    v = internal_fills.dropna(subset=[f'mo_{name}'])
    dollars.append({'horizon': name,
                    'shares': int(v['quantity'].sum()),
                    'edge ($)': round(v['edge'].sum() / 100, 0),
                    'firm markout ($)': round(v[f'mo_{name}'].sum() / 100, 0),
                    'adjusted ($)': round((v['edge'].sum() + v[f'mo_{name}'].sum()) / 100, 0)})
pd.DataFrame(dollars).set_index('horizon')

**Reading the adjusted table (firm view, cents/share):**

- **Wider spreads pay better once risk-adjusted.** The 6¢ bucket reaches +212.5¢/share at
  30 minutes; the 1¢ bucket is the only one that turns negative (−15.35¢). On this day wide
  spreads were compensation, not a warning — the flow arriving at wide spreads was benign.
  Note this is the empirical half of the "wide spreads cut both ways" argument; the
  analytical half (the exit costs half that same wide spread) still stands.
- **Execution edge is a rounding error against inventory P&L.** At 30 minutes, $400 of the
  $21,051 is execution edge — 2%. All the discussion of half-cent tick rounding lives inside
  that 2%.
- **The 1¢ bucket inverts with horizon**: highest edge (0.5¢, a 100% capture of the
  half-spread) but negative long-horizon adjusted P&L, because those are reduce-branch fills
  that close out positions which were still appreciating. Bleeding inventory away therefore
  costs edge but protects markout — the two metrics disagree by construction.
- **Horizon is a policy choice, not a detail**: the same fills read +0.22¢ at 5 seconds and
  +10.42¢ at 30 minutes, a 47× spread. Choose the horizon that matches realised holding time.

---
## Which horizon matches the book? Realised holding time

The adjusted table only means something at the horizon the firm actually carries
inventory for. Two independent estimates, both from the output CSVs:

1. **FIFO share-level pairing** — every closing share is matched against the oldest open
   share, giving a share-weighted average holding time plus its full distribution.
2. **Little's law** — average inventory ÷ one-way flow rate. In steady state this equals
   the mean residence time, so agreement between the two is evidence the day was close to
   steady state rather than one long directional build.

A high turnover ratio is *not* evidence of short holding: a small book that is constantly
replenished still holds each individual share for a while. Turnover measures flow against
inventory; holding time measures residence.

In [ ]:
from collections import deque


def principal_trade_stream(fills_path=FILLS_PATH, firm_path=config.FIRM_TRADES_CSV):
    """(timestamp, firm_signed_qty, fill_row_or_None) for every trade carrying firm risk."""
    ev = []
    f = pd.read_csv(fills_path, parse_dates=['timestamp'])
    for _, r in f[f['capacity'] == 'PRINCIPAL'].iterrows():
        firm_signed = -r['quantity'] if r['side'] == 'BUY' else r['quantity']
        ev.append((r['timestamp'], firm_signed, r))
    h = pd.read_csv(firm_path, parse_dates=['timestamp'])
    for _, r in h.iterrows():
        firm_signed = r['quantity'] if r['side'] == 'BUY' else -r['quantity']
        ev.append((r['timestamp'], firm_signed, None))
    ev.sort(key=lambda x: x[0])
    return ev


def holding_time(ev):
    """FIFO share-weighted holding time + Little's law cross-check + percentiles."""
    lots, durations = deque(), []                       # open lots: (ts, signed qty)
    for ts, d, _ in ev:
        while d and lots and (lots[0][1] > 0) != (d > 0):   # opposite sign closes the lot
            t0, q0 = lots[0]
            m = min(abs(q0), abs(d))
            durations.append(((ts - t0).total_seconds(), m))
            q0 -= m if q0 > 0 else -m
            d -= -m if d < 0 else m
            lots[0] = (t0, q0)
            if q0 == 0:
                lots.popleft()
        if d:
            lots.append((ts, d))

    closed = sum(m for _, m in durations)
    fifo_mean = sum(s * m for s, m in durations) / closed

    span = (ev[-1][0] - ev[0][0]).total_seconds()        # Little's law inputs
    pos = gross = 0
    tw = 0.0
    prev = None
    for ts, d, _ in ev:
        if prev is not None:
            tw += abs(pos) * (ts - prev).total_seconds()
        pos += d
        gross += abs(d)
        prev = ts
    avg_inventory = tw / span
    one_way_flow = gross / 2 / span                      # shares per second
    littles = avg_inventory / one_way_flow

    durations.sort()                                     # share-weighted percentiles
    cum, pct = 0, {}
    for sec, m in durations:
        cum += m
        for p in (25, 50, 75, 90):
            pct.setdefault(p, sec) if cum >= closed * p / 100 else None
    return {'FIFO mean (s)': round(fifo_mean),
            "Little's law (s)": round(littles),
            'p25 (s)': pct[25], 'p50 (s)': pct[50], 'p75 (s)': pct[75], 'p90 (s)': pct[90],
            'avg inventory (sh)': round(avg_inventory),
            'gross traded (sh)': gross}


ev = principal_trade_stream()
pd.Series(holding_time(ev))   # FIFO and Little's law agree -> steady-state turnover

In [ ]:
# Is the 1c bucket entirely risk-reducing? (it should be: at 1c only the reduce
# branch can fill, capped at min(order.remaining, abs(position)) and priced at the touch)
pos, cls = 0, []
for ts, firm_signed, row in ev:
    if row is not None and row['venue'] == 'INTERNAL':
        spread_c = round((row['nbbo_ask'] - row['nbbo_bid']) * 100)
        if spread_c == 1:
            at_touch = (round(row['price'] * 100) ==
                        round((row['nbbo_ask'] if row['side'] == 'BUY' else row['nbbo_bid']) * 100))
            effect = ('flat-start' if pos == 0 else
                      'reduce' if (pos > 0) != (firm_signed > 0) else 'increase')
            cls.append({'effect': effect, 'at_touch': at_touch, 'qty': row['quantity'],
                        'flips_sign': abs(pos) < row['quantity']})
    pos += firm_signed

c = pd.DataFrame(cls)
print(f"1c INTERNAL fills: {len(c)} fills / {c['qty'].sum():,} shares")
print(f"  all at the touch : {c['at_touch'].all()}")
print(f"  any sign flip    : {c['flips_sign'].any()}")
c.groupby('effect')['qty'].agg(fills='count', shares='sum')